## DocuScope assessment

##### 1 - View DocuScope tags in an email

Loads the `en_docusco_spacy` model and the study emails from `corpus.ipynb`, and highlights DocuScope categories in a single email body. Call `show_email(msgid2)` with any email ID.

In [ ]:
import itertools
import pandas as pd
import spacy
from spacy import displacy
from pathlib import Path

# Repository root (the folder with _quarto.yml), found from this notebook's folder
root = Path.cwd()
while not (root / '_quarto.yml').exists():
    root = root.parent
here = root / 'coding' / 'rhetoric' / 'tests' / 'docuscope'

nlp = spacy.load('en_docusco_spacy')
emails_study = pd.read_parquet(root / 'corpus' / 'emails_study.parquet')

palette = ['#f8d7e3', '#d6e6fb', '#fde2c4', '#e5dcf7', '#d5f0dc', '#fdf3c4']
labels = nlp.get_pipe('ner').labels
colors = dict(zip(labels, itertools.cycle(palette)))

def show_email(msgid2):
    """Show one email's body with DocuScope categories highlighted."""
    body = emails_study.loc[emails_study['msgid2'] == msgid2, 'body'].iloc[0]
    displacy.render(nlp(body), style='ent', jupyter=True, options={'colors': colors})

# Example: the first study email
show_email(emails_study['msgid2'].iloc[0])

##### 2 - Export 30 tagged emails to HTML

Draws 30 emails at random from the pool (`in_pool`, from `corpus.ipynb` block 3: 300 to 2,000 characters, no credential-like text), highlights their DocuScope categories, and saves them as one HTML page, `docuscope_sample_30.html`, beside this notebook.

In [ ]:
import html
from collections import Counter

# Pick 30 random emails from the pool
pool = emails_study[emails_study['in_pool']]
sample = pool.sample(30, random_state=42)

# Rhetorical categories, in a provisional working grouping (not from Brown and Laudenbach 2022)
rhetoric_groups = {
    "Interaction": ["Interactive", "Facilitate", "MetadiscourseInteractive", "Inquiry"],
    "Stance":      ["ConfidenceHigh", "ConfidenceHedged", "ForceStressed",
                    "Uncertainty", "Contingent", "Responsibility"],
    "Evaluation":  ["Positive", "Negative"],
    "Planning":    ["Future", "Updates", "Narrative", "Strategic"],
    "Self":        ["FirstPerson"],
    "Cohesion":    ["MetadiscourseCohesive", "Reasoning"],
}

# Content: content-bearing tags
content_groups = {
    "Topics":      ["InformationTopics", "AcademicTerms", "PublicTerms"],
    "Reporting":   ["InformationReportVerbs", "InformationExposition", "InformationChange"],
    "Places":      ["InformationPlace"],
    "Description": ["Description"],
}

rhetoric_cats  = {c for cats in rhetoric_groups.values() for c in cats}
content_cats = {c for cats in content_groups.values() for c in cats}

def phrases_by_cat(doc):
    """Map each tag to its distinct phrases (in order of appearance) and total count."""
    out = {}
    for ent in doc.ents:
        p = " ".join(ent.text.split())
        phrases, n = out.get(ent.label_, ([], 0))
        if p.lower() not in [x.lower() for x in phrases]:
            phrases.append(p)
        out[ent.label_] = (phrases, n + 1)
    return out

def group_block(found, groups):
    """HTML lines: one per group, listing each category with count and phrases."""
    lines = []
    for gname, cats in groups.items():
        items = []
        for cat in cats:
            if cat in found:
                phrases, n = found[cat]
                items.append(f"<i>{cat}</i> ({n}): " + html.escape("; ".join(phrases)))
        if items:
            lines.append(f"<b>{gname}</b> — " + " &nbsp;|&nbsp; ".join(items))
    return "<br>".join(lines) if lines else "<i>none</i>"

# Build each email and tally tags across the sample
totals = Counter()
parts = []
for i, row in enumerate(sample.itertuples(), 1):
    doc = nlp(row.body)
    totals.update(ent.label_ for ent in doc.ents)
    found = phrases_by_cat(doc)
    tagged = displacy.render(doc, style='ent', jupyter=False, page=False,
                             options={'colors': colors})
    parts.append(f"""
    <div class="email">
      <h2>Email {i} of 30</h2>
      <p class="meta">
        <b>msgid2:</b> {row.msgid2} &nbsp;|&nbsp; <b>Study:</b> {html.escape(str(row.msg_citfid))}<br>
        <b>From:</b> {html.escape(str(row._3))}<br>
        <b>Subject:</b> {html.escape(str(row.subject))}<br>
        <b>Length:</b> {len(row.body)} characters
      </p>
      <div class="box how"><div class="label">Rhetorical</div>{group_block(found, rhetoric_groups)}</div>
      <div class="box what"><div class="label">Content</div>{group_block(found, content_groups)}</div>
      {tagged}
    </div>""")

# ---- Overall summary (top of page) ----
n_all   = sum(totals.values())
n_how   = sum(v for k, v in totals.items() if k in rhetoric_cats)
n_what  = sum(v for k, v in totals.items() if k in content_cats)
n_other = n_all - n_how - n_what
pct = lambda n: 100 * n / n_all if n_all else 0

def group_rows(groups, cls):
    rows = []
    for gname, cats in groups.items():
        g_n = sum(totals.get(c, 0) for c in cats)
        detail = ", ".join(f"{c} {totals.get(c, 0)}" for c in cats if totals.get(c, 0))
        rows.append(f"<tr class='{cls}'><td>{gname}</td><td>{g_n}</td>"
                    f"<td>{pct(g_n):.1f}%</td><td>{detail}</td></tr>")
    return "".join(rows)

other_detail = ", ".join(f"{k} {v}" for k, v in totals.most_common()
                         if k not in rhetoric_cats and k not in content_cats)

summary = f"""
<div class="summary">
  <h2>Overall tag split (n = 30 emails, {n_all} tags)</h2>
  <div class="bar">
    <div class="seg how-bg"   style="width:{pct(n_how):.1f}%">Rhetorical {pct(n_how):.0f}%</div>
    <div class="seg what-bg"  style="width:{pct(n_what):.1f}%">Content {pct(n_what):.0f}%</div>
    <div class="seg other-bg" style="width:{pct(n_other):.1f}%">Other {pct(n_other):.0f}%</div>
  </div>
  <table>
    <tr><th>Group</th><th>Tags</th><th>% of all</th><th>Categories</th></tr>
    <tr class="how head"><td><b>Rhetorical total</b></td><td><b>{n_how}</b></td><td><b>{pct(n_how):.1f}%</b></td><td></td></tr>
    {group_rows(rhetoric_groups, 'how')}
    <tr class="what head"><td><b>Content total</b></td><td><b>{n_what}</b></td><td><b>{pct(n_what):.1f}%</b></td><td></td></tr>
    {group_rows(content_groups, 'what')}
    <tr class="other head"><td><b>Other</b></td><td><b>{n_other}</b></td><td><b>{pct(n_other):.1f}%</b></td><td>{other_detail}</td></tr>
  </table>
</div>"""

print(f"Rhetorical {n_how} ({pct(n_how):.1f}%) | Content {n_what} ({pct(n_what):.1f}%) | Other {n_other} ({pct(n_other):.1f}%)")

page = f"""<!doctype html>
<html><head><meta charset="utf-8"><title>DocuScope sample</title>
<style>
  body {{ font-family: sans-serif; max-width: 900px; margin: 40px auto; padding: 0 16px; }}
  .email {{ border-top: 2px solid #ccc; padding: 24px 0; }}
  .meta {{ color: #555; font-size: 0.9em; }}
  .box {{ padding: 8px 12px; margin: 8px 0; font-size: 0.85em; line-height: 1.6; border-left: 4px solid; }}
  .box .label {{ font-weight: bold; letter-spacing: 0.05em; margin-bottom: 4px; }}
  .how  {{ background: #f3f0fa; border-color: #8a75c9; }}
  .what {{ background: #eef6f1; border-color: #5a9e74; }}
  .summary {{ margin: 24px 0 32px; }}
  .bar {{ display: flex; height: 32px; border-radius: 6px; overflow: hidden; margin: 12px 0; }}
  .seg {{ color: #fff; font-size: 0.85em; font-weight: bold; display: flex;
          align-items: center; justify-content: center; white-space: nowrap; overflow: hidden; }}
  .how-bg {{ background: #8a75c9; }}  .what-bg {{ background: #5a9e74; }}  .other-bg {{ background: #aaa; }}
  table {{ border-collapse: collapse; width: 100%; font-size: 0.85em; }}
  th, td {{ text-align: left; padding: 5px 8px; border-bottom: 1px solid #e5e5e5; vertical-align: top; }}
  tr.how.head {{ background: #e6e0f5; }}  tr.what.head {{ background: #dcefe3; }}  tr.other.head {{ background: #eee; }}
</style></head>
<body><h1>DocuScope tags: 30 sampled study emails</h1>
<p class="meta">Sample: emails of 300–2000 characters, credential-like emails excluded,
random_state=42.<br>
Rhetorical = stance, interaction and planning categories, in a provisional working grouping.
Content = content-bearing categories. Other = tags in neither list. Counts are tag occurrences.</p>
{summary}
{''.join(parts)}
</body></html>"""

out_path = here / 'docuscope_sample_30.html'
with open(out_path, 'w', encoding='utf-8') as f:
    f.write(page)
print("Saved to:", out_path)

##### 3 - Detect likely tagging errors

Leaves out automated notifications (senders listed in `automated_senders`) and emails without zones, reads the remaining pool emails as `rhetoric.ipynb` block 1 parsed them, and gives each tag the zone of the line it starts on, from `email_zones.parquet` (`corpus.ipynb` block 5). Checks that the lines in the zone file match the email bodies. Flags rhetorical tags in authored text that match an error rule; each rule names the categories it targets. Rules whose matches count as errors without coding (`fixed_rules`) go straight to block 5; the others get a manual check. Lists capitalized runs of tags as candidate names and adjacent tags from different categories as candidate formulas, for issues that span more than one tag. Draws a stratified coding sample (up to 30 tags per checked rule, 10 per domain noun, and 20 per rhetorical category among tags that match no rule), saves it to `docuscope_coding_sample.csv`, and writes a coding tool for it to `docuscope_coding_tool.html`. All patterns sit at the top of the cell.

In [ ]:
import os
import re
import json
import numpy as np
from spacy.tokens import DocBin

# Print tables as text so the full report is one copyable output
def show(df):
    print(df.to_string(max_colwidth=90))

# ---- Settings (edit and rerun; all matching is case-insensitive) ----

# Zones of every line, from corpus.ipynb block 5
zones_path = root / 'corpus' / 'email_zones.parquet'

# Senders whose emails are automated notifications; these emails are left out
automated_senders = [r"docusign", r"no-?reply", r"mailer-daemon", r"postmaster", r"microsoft"]

# Multiword names that DocuScope splits into separate tags
name_patterns = [
    r"COVID-19 Immunity Task Force", r"Immunity Task Force",
    r"Groupe de travail sur l['’]immunit[ée]",
    r"Unity Health( Toronto)?", r"McGill University Health Cent(re|er)",
    r"Alberta Health Services", r"Saskatchewan Health Authority", r"Research Institute",
]

# Fixed formulas: (pattern, categories whose tags inside the formula are flagged)
formula_rules = {
    'dont_hesitate': (r"(do not|don['’]?t) hesitate to", {'Uncertainty', 'Negative', 'ForceStressed'}),
    'let_know':      (r"(let|letting) (me|us) know", {'ConfidenceHigh'}),
    'on_my_end':     (r"on (my|our|your|their) end", {'Narrative'}),
    'terms_cond':    (r"terms and conditions", {'Contingent'}),
    'best_of':       (r"to the best of", {'Positive'}),
    'have_a_day':    (r"have a (great|good|nice|wonderful) (day|weekend|week)", {'Strategic', 'Narrative'}),
}

# Nouns from the data sharing domain (tag text, lowercased)
domain_nouns = {
    'agreement', 'agreements', 'consent', 'cohort', 'cohorts', 'assay', 'assays',
    'immunity', 'health', 'research', 'results', 'analysis', 'analyses', 'summary',
    'harmonize', 'harmonization', 'data preparation', 'collected', 'collect', 'access',
}

# Rules whose matches are treated as errors without coding; blocks 4 and 5 apply them directly,
# so they stay out of the coding sample
fixed_rules = {'date_month', 'time_am_pm', 'name', 'french_line', 'terms_cond'}

# French function words, for flagging French lines
fr_words = {'le', 'la', 'les', 'des', 'du', 'de', 'et', 'vous', 'nous', 'est', 'pour',
            'avec', 'sur', 'dans', 'une', 'au', 'aux', 'ce', 'qui', 'que', 'pas', 'à'}
en_words = {'the', 'and', 'to', 'of', 'you', 'is', 'for', 'we', 'in', 'on', 'with', 'this'}

# ---- Helpers ----

def tok(doc, i):
    return doc[i].text.lower() if 0 <= i < len(doc) else ''

# Month names, full or abbreviated
months = (r"jan(uary)?|feb(ruary)?|mar(ch)?|apr(il)?|may|june?|july?|aug(ust)?|sept?(ember)?"
          r"|oct(ober)?|nov(ember)?|dec(ember)?")

# Rules on the tag and its neighbouring tokens: (test, categories flagged; None = any category)
token_rules = {
    'date_month':    (lambda e: re.fullmatch(months, e.text.lower().rstrip('.'))
                                and (re.match(r'\d', tok(e.doc, e.end)) or re.match(r'\d', tok(e.doc, e.start - 1)))
                                and e.label_ != 'Narrative',
                      None),
    'time_am_pm':    (lambda e: re.fullmatch(r'([\d:.]+\s*)?(a\.?m\.?|p\.?m\.?)', e.text.lower())
                                and (re.search(r'\d', e.text) or re.search(r'\d', tok(e.doc, e.start - 1))),
                      {'FirstPerson'}),
    'negated_modal': (lambda e: e.text.lower().split()[-1] in ('can', 'ca', 'do', 'does', 'did')
                                and tok(e.doc, e.end) in ('not', "n't", 'n’t'),
                      {'ForceStressed', 'ConfidenceHedged'}),
    'polite_would':  (lambda e: re.fullmatch(r'((i|we|you|they|she|he|it) )?would( be)?', e.text.lower()),
                      {'Narrative'}),
    'domain_noun':   (lambda e: e.text.lower() in domain_nouns, None),
}

def line_spans(text):
    pos = 0
    for line in text.split('\n'):
        yield pos, pos + len(line), line
        pos += len(line) + 1

def french_lines(text):
    out = []
    for s, e, line in line_spans(text):
        words = re.findall(r"[a-zà-ÿ]+", line.lower())
        fr, en = sum(w in fr_words for w in words), sum(w in en_words for w in words)
        if fr >= 3 and fr > en:
            out.append((s, e))
    return out

def overlaps(a, b, ranges):
    return any(a < e and s < b for s, e in ranges)

# ---- Emails: leave out automated notifications and emails without zones ----
zones = pd.read_parquet(zones_path, columns=['msgid2', 'start', 'end', 'text', 'zone'])
zones['zone'] = zones['zone'].fillna('empty')
zones['msgid2'] = zones['msgid2'].astype(str)
# Pool from corpus.ipynb block 3, parsed by rhetoric.ipynb block 1
pool = emails_study[emails_study['in_pool']]
docs = {d.user_data['msgid2']: d for d in
        DocBin().from_disk(root / 'coding' / 'rhetoric' / 'docs.spacy').get_docs(nlp.vocab)}
is_auto = pool['from'].astype(str).str.contains('|'.join(automated_senders), case=False, regex=True)
has_zones = pool['msgid2'].astype(str).isin(zones['msgid2'])
print("Automated emails left out, by sender:")
show(pool.loc[is_auto, 'from'].value_counts().rename('emails').to_frame())
print(f"Emails without zones, left out: {(~is_auto & ~has_zones).sum():,}")
pool9 = pool[~is_auto & has_zones]

# Line starts, ends and zones of each email, in order
zone_lines = {m: (g['start'].to_numpy(), g['end'].to_numpy(), g['zone'].to_numpy(), g['text'].astype(str).to_numpy())
              for m, g in zones.sort_values(['msgid2', 'start']).groupby('msgid2')}

def tag_zone(starts, ends, zs, a):
    """Zone of the line the tag starts on."""
    i = np.searchsorted(starts, a, side='right') - 1
    return zs[i] if i >= 0 and a <= ends[i] else 'none'

# ---- Parse and label every tag ----

rows, mismatched, checked = [], [], 0
for msgid2, body in zip(pool9['msgid2'], pool9['body'].astype(str)):
    doc = docs[str(msgid2)]
    starts, ends, zs, texts = zone_lines[str(msgid2)]
    # Check that the zone file's lines match this body
    for s, e, t in zip(starts, ends, texts):
        checked += 1
        if body[s:e].strip() != t.strip():
            mismatched.append(dict(msgid2=msgid2, start=s, zone_file=t, body=body[s:e]))
    fr = french_lines(body)
    names = [(m.start(), m.end()) for p in name_patterns for m in re.finditer(p, body, re.I)]
    formulas = {k: [(m.start(), m.end()) for m in re.finditer(p, body, re.I)]
                for k, (p, cats) in formula_rules.items()}
    for e in doc.ents:
        a, b = e.start_char, e.end_char
        rules = [k for k, (f, cats) in token_rules.items() if (cats is None or e.label_ in cats) and f(e)]
        rules += [k for k, r in formulas.items() if e.label_ in formula_rules[k][1] and overlaps(a, b, r)]
        if overlaps(a, b, names):
            rules.append('name')
        if overlaps(a, b, fr):
            rules.append('french_line')
        rows.append(dict(msgid2=msgid2, tag=e.label_, phrase=' '.join(e.text.split()),
                         start=a, end=b, tok_start=e.start, tok_end=e.end,
                         zone=tag_zone(starts, ends, zs, a), rules=';'.join(rules),
                         left=' '.join(body[max(0, a - 60):a].split()),
                         right=' '.join(body[b:b + 60].split())))

tags9 = pd.DataFrame(rows)
how9 = tags9[tags9['tag'].isin(rhetoric_cats)].copy()

# ---- Check: zone file lines against email bodies ----
print(f"\nEmails: {len(pool9):,} | rhetorical tags: {len(how9):,}")
print(f"Zone-file lines that do not match the email body: {len(mismatched):,} of {checked:,}")
if mismatched:
    show(pd.DataFrame(mismatched).head(10))

# ---- Where rhetorical tags fall, per category (% of the category's tags) ----
zone_tab = pd.crosstab(how9['tag'], how9['zone'])
zone_tab = zone_tab.div(zone_tab.sum(axis=1), axis=0).mul(100)
zone_tab.insert(0, 'tags', how9['tag'].value_counts())
print("\nWhere rhetorical tags fall (% of each category's tags):")
show(zone_tab.sort_values('tags', ascending=False).round(1))

# ---- Rule matches (authored text only) ----
auth = how9[how9['zone'] == 'authored']
rule_rows = auth.assign(rule=auth['rules'].str.split(';')).explode('rule')
rule_rows = rule_rows[rule_rows['rule'] != '']
rule_tab = (rule_rows.groupby('rule')
            .agg(tags=('tag', 'size'),
                 categories=('tag', lambda s: '; '.join(f"{k} {v}" for k, v in s.value_counts().head(4).items()))))
rule_tab.insert(0, 'kind', ['fixed' if r in fixed_rules else 'checked' for r in rule_tab.index])
print("\nRule matches in authored text (fixed = applied without coding; checked = coded first):")
show(rule_tab.sort_values(['kind', 'tags'], ascending=[True, False]))

# Authored rhetorical tags per category matched by any rule
matched = auth.assign(any_rule=auth['rules'] != '')
match_tab = (matched.groupby('tag')['any_rule'].agg(['size', 'sum'])
             .rename(columns={'size': 'authored_tags', 'sum': 'matched_by_rule'}))
match_tab['pct_matched'] = 100 * match_tab['matched_by_rule'] / match_tab['authored_tags']
print("\nAuthored rhetorical tags matched by any rule, per category:")
show(match_tab.sort_values('authored_tags', ascending=False).round(1))

# ---- Save rhetorical tags with their zones and rule matches, for analyses after this block ----
# One row per rhetorical tag. Holds phrases from the emails, so it stays in data/, which git ignores.
meta_cols = ['from', 'msg_citfid']   # sender and study ID
how_out = how9[['msgid2', 'tag', 'phrase', 'start', 'end', 'zone', 'rules', 'left', 'right']].merge(
    pool9[['msgid2'] + meta_cols], on='msgid2', how='left')
tags_path = here / 'rhetoric_tags.parquet'
how_out.to_parquet(tags_path, index=False)
print(f"\nrhetorical tags saved: {len(how_out):,} rows from {how_out['msgid2'].nunique():,} emails")
print(f"Columns: {list(how_out.columns)}")
print(f"Saved to: {tags_path}")

# ---- Tags that run together (issues spanning more than one tag) ----
# A run is a sequence of tags in authored text with no untagged tokens between them
tags9 = tags9.sort_values(['msgid2', 'tok_start']).reset_index(drop=True)
new_run = ((tags9['msgid2'] != tags9['msgid2'].shift())
           | (tags9['tok_start'] != tags9['tok_end'].shift())
           | (tags9['zone'] != 'authored'))
tags9['run'] = new_run.cumsum()
in_text = tags9[tags9['zone'] == 'authored']

# Candidate names: runs of two or more capitalized tags, at least one of them HOW
cap = in_text['phrase'].str[:1].str.isupper()
name_runs = (in_text[cap].groupby('run')
             .agg(phrase=('phrase', ' '.join), tags=('tag', ' + '.join), n_tags=('tag', 'size'),
                  has_how=('tag', lambda t: t.isin(rhetoric_cats).any()),
                  msgid2=('msgid2', 'first'), left=('left', 'first'), right=('right', 'last')))
name_runs = name_runs[(name_runs['n_tags'] > 1) & name_runs['has_how']]
print("\nCandidate names (capitalized runs of 2+ tags), with one example context each:")
name_tab = (name_runs.groupby(['phrase', 'tags'])
            .agg(n=('msgid2', 'size'), msgid2=('msgid2', 'first'),
                 left=('left', 'first'), right=('right', 'first'))
            .reset_index().sort_values('n', ascending=False).head(30))
show(name_tab)

# Candidate formulas: adjacent pairs of tags from different categories, at least one of them HOW
nxt = in_text.shift(-1)
pairs = in_text[(nxt['run'] == in_text['run']) & (nxt['tag'] != in_text['tag'])
                & (in_text['tag'].isin(rhetoric_cats) | nxt['tag'].isin(rhetoric_cats))]
pairs = pd.DataFrame({
    'phrase': (pairs['phrase'] + ' ' + nxt.loc[pairs.index, 'phrase']).str.lower(),
    'tags': pairs['tag'] + ' + ' + nxt.loc[pairs.index, 'tag']})
print("\nCandidate formulas (adjacent tags from different categories):")
show(pairs.groupby(['phrase', 'tags']).size().rename('n').reset_index()
        .sort_values('n', ascending=False).head(30))

# ---- Coding sample for manual review (stratified) ----
# Strata: each checked rule (up to 30 tags), each noun of domain_noun (up to 10 tags),
# and each rhetorical category among tags that match no rule (up to 20 tags).
# stratum_size is the number of authored tags in the stratum, for weighting results back in block 5.
n_per_rule, n_per_noun, n_per_category = 30, 10, 20
draws = []
for r, g in rule_rows[~rule_rows['rule'].isin(fixed_rules | {'domain_noun'})].groupby('rule'):
    draws.append(g.sample(min(len(g), n_per_rule), random_state=42)
                 .assign(sample=f"rule:{r}", stratum=f"rule:{r}", stratum_size=len(g)))
nouns = rule_rows[rule_rows['rule'] == 'domain_noun'].assign(noun=lambda d: d['phrase'].str.lower())
for n, g in nouns.groupby('noun'):
    draws.append(g.sample(min(len(g), n_per_noun), random_state=42)
                 .assign(sample='rule:domain_noun', stratum=f"rule:domain_noun:{n}", stratum_size=len(g)))
for c, g in auth[auth['rules'] == ''].groupby('tag'):
    draws.append(g.sample(min(len(g), n_per_category), random_state=42)
                 .assign(sample='no_rule', stratum=f"no_rule:{c}", stratum_size=len(g)))
coding = pd.concat(draws)
coding = coding[~coding.index.duplicated(keep='first')]   # a tag matched by two rules appears once
coding = coding[['sample', 'stratum', 'stratum_size', 'msgid2', 'tag', 'phrase', 'left', 'right', 'rules']]
coding.insert(0, 'id', range(1, len(coding) + 1))
coding['coder1'] = ''
coding['coder2'] = ''
coding['notes'] = ''
coding_path = here / 'docuscope_coding_sample.csv'
coding.to_csv(coding_path, index=False)
print("\nCoding sample by sample type:")
show(coding.groupby('sample').agg(strata=('stratum', 'nunique'), tags=('id', 'size')))
print(f"Coding sample: {len(coding):,} tags saved to: {coding_path}")

# Shown on each card: what the category means (DocuScope Full Dictionary documentation,
# https://browndw.github.io/docuscope-docs/tagsets/docuscope-fd-tags.html) and why the card is there
category_descriptions = {
    'Interactive': ("Addresses the reader directly: second person, questions, requests.", "can you; thank you for; please see"),
    'Facilitate': ("Guides or helps the reader through tasks and actions.", "let me; worth a try; I would suggest"),
    'MetadiscourseInteractive': ("Reader-facing markers that engage the audience in the exchange.", "I agree; let's talk; by the way"),
    'Inquiry': ("Points to questions, investigation or searching.", "find out; wondering if"),
    'ConfidenceHigh': ("Presents a claim as certain.", "most likely; ensure that; obviously"),
    'ConfidenceHedged': ("Presents a claim as uncertain or tentative.", "maybe; it seems that; tends to get"),
    'ForceStressed': ("Emphatic wording: intensifiers, superlatives, insistence.", "really good; necessary"),
    'Uncertainty': ("Signals that the writer's level of confidence is unknown.", "kind of; I have no idea; for some reason"),
    'Contingent': ("Conditions that depend on circumstances in the world.", "subject to; if possible; hypothetically"),
    'Responsibility': ("Duties, requirements and obligations.", "supposed to; requirements; obligations"),
    'Positive': ("Positive actions, emotions, relations and values.", "thanks; approval; agreement"),
    'Negative': ("Negative actions, emotions, relations and values.", "does not; sorry for; problems"),
    'Future': ("Future actions, states or desires.", "will be; hope to; expected changes"),
    'Updates': ("Signals new information for a reader who is seeking it.", "already; a new; now that"),
    'Narrative': ("Events, people and descriptions extending in time.", "today; tomorrow; this weekend"),
    'Strategic': ("Plans, goals, strategies and seeking advantage.", "plan; trying to; strategy"),
    'FirstPerson': ("References to the writer in the first person.", "I; we have been; as soon as I"),
    'MetadiscourseCohesive': ("Markers that move through the text and link its parts.", "or; but; also"),
    'Reasoning': ("Inferences about cause, consequence, concession or conclusion.", "because; therefore; even if"),
}
rule_descriptions = {
    'domain_noun': "A noun from the data sharing domain (agreement, consent, cohort, research, results and others). "
                   "Flagged because DocuScope may read a document or concept as evaluation or stance.",
    'polite_would': "'would' or 'you would' tagged Narrative. Flagged because polite requests "
                    "('if you would please') describe no event in time.",
    'let_know': "'know' in 'let me/us know' tagged ConfidenceHigh. Flagged because the phrase is a request, "
                "not a claim of certainty.",
    'dont_hesitate': "A word in 'don't hesitate to' tagged Uncertainty, Negative or ForceStressed. Flagged because "
                     "the formula invites contact and expresses no doubt or negativity.",
    'negated_modal': "'do', 'does', 'did' or 'can' before 'not' or 'n't', tagged ForceStressed or ConfidenceHedged. "
                     "Flagged because the word may only be part of a negation.",
    'have_a_day': "A word in 'have a great day/weekend' tagged Strategic or Narrative. Flagged because the formula "
                  "is a farewell, not a plan or an event.",
    'on_my_end': "'end' in 'on my/our/your end' tagged Narrative. Flagged because 'end' means 'side' here, "
                 "not a point in time.",
    'best_of': "'the best of' in 'to the best of your knowledge' tagged Positive. Flagged because the formula "
               "qualifies an answer and evaluates nothing.",
    'date_month': "Fixed rule: a month name next to a day number, tagged outside Narrative (mostly 'May' as hedging).",
    'time_am_pm': "Fixed rule: 'am' after a clock time, tagged FirstPerson.",
    'name': "Fixed rule: part of a multiword name such as COVID-19 Immunity Task Force.",
    'french_line': "Fixed rule: a line of French text, tagged by an English-only model.",
    'terms_cond': "Fixed rule: 'conditions' in 'terms and conditions', tagged Contingent.",
    'no_rule': "No rule matched. A random tag from authored text in this category, used to estimate how often "
               "DocuScope is wrong where no rule applies.",
}

# ---- Coding tool: one HTML page with the sample embedded; each coder exports their own CSV ----
coding_html = r'''<!doctype html>
<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>DocuScope coding</title>
<style>
:root { --bg:#f7f7f5; --card:#fff; --ink:#1f2328; --muted:#6b7075; --line:#e2e2de; --mark:#ffe08a;
        --ok:#2f8f5b; --bad:#c2453d; --unsure:#b7871a; --accent:#4b5fc4; }
@media (prefers-color-scheme: dark) { :root { --bg:#17181a; --card:#212327; --ink:#e8e8e6; --muted:#9a9ea3;
        --line:#33363b; --mark:#7a5f12; --ok:#4fbf83; --bad:#e46a62; --unsure:#e0b04a; --accent:#8b9cf0; } }
* { box-sizing:border-box; }
body { margin:0; background:var(--bg); color:var(--ink); font:16px/1.5 system-ui, -apple-system, sans-serif; }
header { display:flex; flex-wrap:wrap; gap:10px; align-items:center; padding:12px 16px; border-bottom:1px solid var(--line); background:var(--card); position:sticky; top:0; }
header h1 { font-size:16px; margin:0 8px 0 0; }
select, input[type=text], button { font:inherit; font-size:14px; padding:5px 9px; border:1px solid var(--line); border-radius:6px; background:var(--bg); color:var(--ink); }
button { cursor:pointer; }
button.primary { background:var(--accent); color:#fff; border-color:var(--accent); }
.spacer { flex:1; }
.progress { height:4px; background:var(--line); }
.progress div { height:100%; background:var(--accent); width:0; transition:width .2s; }
main { max-width:860px; margin:24px auto; padding:0 16px; }
.card { background:var(--card); border:1px solid var(--line); border-radius:10px; padding:22px; }
.meta { display:flex; flex-wrap:wrap; gap:8px; color:var(--muted); font-size:13px; margin-bottom:14px; }
.pill { border:1px solid var(--line); border-radius:999px; padding:1px 9px; }
.tag { font-size:26px; font-weight:650; margin:0 0 4px; }
.phrase { color:var(--muted); margin-bottom:12px; }
.def { font-size:15px; margin:0 0 4px; }
.def .eg { color:var(--muted); }
.why { font-size:14px; border-left:3px solid var(--accent); padding:6px 12px; margin:10px 0 16px; background:var(--bg); border-radius:0 6px 6px 0; }
.why b { font-weight:600; }
.why div + div { margin-top:4px; }
.context { font-size:19px; line-height:1.7; padding:14px 16px; background:var(--bg); border-radius:8px; }
.context mark { background:var(--mark); color:inherit; padding:1px 3px; border-radius:3px; font-weight:600; }
.codes { display:grid; grid-template-columns:repeat(3,1fr); gap:10px; margin-top:18px; }
.codes button { padding:12px; font-size:15px; border-width:2px; }
.codes button[data-code=correct].on { background:var(--ok); border-color:var(--ok); color:#fff; }
.codes button[data-code=incorrect].on { background:var(--bad); border-color:var(--bad); color:#fff; }
.codes button[data-code=unsure].on { background:var(--unsure); border-color:var(--unsure); color:#fff; }
.note { width:100%; margin-top:12px; }
.nav { display:flex; justify-content:space-between; align-items:center; margin-top:14px; color:var(--muted); font-size:14px; }
.keys { color:var(--muted); font-size:13px; margin-top:16px; text-align:center; }
kbd { border:1px solid var(--line); border-bottom-width:2px; border-radius:4px; padding:0 5px; font-size:12px; background:var(--card); }
.empty { text-align:center; color:var(--muted); padding:40px; }
#about { max-width:860px; margin:20px auto 0; padding:0 16px; }
#about .card { font-size:15px; }
#about h2 { font-size:18px; margin:0 0 8px; }
#about h3 { font-size:15px; margin:18px 0 4px; }
#about p, #about li { margin:4px 0; }
#about table { border-collapse:collapse; font-size:14px; margin-top:6px; }
#about td { border-top:1px solid var(--line); padding:4px 14px 4px 0; vertical-align:top; }
#about[hidden] { display:none; }
@media (max-width:560px) { .codes { grid-template-columns:1fr; } .tag { font-size:22px; } .context { font-size:17px; } }
</style></head>
<body>
<header>
  <h1>DocuScope coding</h1>
  <button id="aboutBtn" aria-expanded="false">About</button>
  <label>Coder <select id="coder"><option value="">choose…</option><option>coder1</option><option>coder2</option></select></label>
  <label>Show <select id="filter"></select></label>
  <label><input type="checkbox" id="uncoded"> uncoded only</label>
  <span class="spacer"></span>
  <span id="count"></span>
  <button id="import">Import</button><input type="file" id="file" accept=".csv" hidden>
  <button id="export" class="primary">Export CSV</button>
</header>
<div class="progress"><div id="bar"></div></div>
<section id="about" hidden><div class="card">
  <h2>About this page</h2>
  <p>This page is the manual check in block 3 of <code>docuscope.ipynb</code>. It shows a stratified sample of rhetorical tags that DocuScope assigned to authored text in CITF study emails, one at a time, so that two coders can judge whether each tag is accurate.</p>
  <h3>Why we code these tags</h3>
  <p>The coding does two jobs. It tests the error rules in block 3: each checked rule claims that tags like the ones it matches are DocuScope errors, and its sample shows whether that claim holds before block 5 relabels every match in the corpus. It also estimates how often DocuScope is wrong where no rule applies, for each rhetorical category, which shows which categories are reliable enough to use.</p>
  <p>The codes may later train a model that predicts whether any DocuScope tag is accurate in its context. Whether that is worth doing, and for which categories, depends on the error rates this sample reveals.</p>
  <h3>What a card shows</h3>
  <p>The DocuScope category with a short description and examples, the tagged phrase, why the tag is in the sample (the rule it matched and what that rule claims, or "No rule"), and about 60 characters of text on each side with the phrase highlighted. The pills give the stratum, the number of authored tags in that stratum, and the <code>msgid2</code>. Category descriptions paraphrase the <a href="https://browndw.github.io/docuscope-docs/tagsets/docuscope-fd-tags.html">DocuScope Full Dictionary tag documentation</a>, the tagset of the <code>en_docusco_spacy</code> model.</p>
  <h3>How to code</h3>
  <ul>
    <li><b>correct</b>: the category's meaning fits what the phrase does in that sentence.</li>
    <li><b>incorrect</b>: it does not fit (for example "know" as ConfidenceHigh in "please let me know").</li>
    <li><b>unsure</b>: the context is too short or the case is genuinely ambiguous. Add a note.</li>
  </ul>
  <p>Code alone and without seeing the other coder's file, so that agreement between coders can be measured.</p>
  <h3>How the sample is drawn</h3>
  <p>The sample is stratified, with random draws inside each stratum (<code>random_state=<span id="seed"></span></code>):</p>
  <ul>
    <li><code>rule:&lt;name&gt;</code>: up to <span id="nPerRule"></span> tags matched by each checked rule.</li>
    <li><code>rule:domain_noun:&lt;noun&gt;</code>: up to <span id="nPerNoun"></span> tags for each domain noun, since different nouns may behave differently.</li>
    <li><code>no_rule:&lt;category&gt;</code>: up to <span id="nPerCat"></span> tags per rhetorical category among tags that match no rule.</li>
  </ul>
  <p>Rules whose matches are treated as errors without coding (<span id="fixedRules"></span>) are applied in block 5. A tag matched by two rules appears once. Each card records its stratum size, so block 5 can weight the coded results back to the whole corpus.</p>
  <table id="sampleTable"></table>
  <h3>Saving and exporting</h3>
  <p>Codes save automatically in this browser, separately for each coder. <b>Export CSV</b> downloads <code>docuscope_coding_coder1.csv</code> or <code>docuscope_coding_coder2.csv</code> with one row per tag: <code>id</code>, sample, stratum, stratum size, <code>msgid2</code>, category, phrase, rules, the code and the note. Save that file in <code>coding/rhetoric/tests/docuscope</code> for block 5. <b>Import</b> reloads an exported file, to resume on another computer or after clearing the browser.</p>
  <h3>Data</h3>
  <p>The page contains excerpts of CITF emails. It works offline and sends nothing anywhere. Keep it and the exported files on storage approved for the study data.</p>
  <p id="generated"></p>
</div></section>
<main id="main"></main>
<p class="keys"><kbd>1</kbd> correct &nbsp; <kbd>2</kbd> incorrect &nbsp; <kbd>3</kbd> unsure &nbsp; <kbd>←</kbd> <kbd>→</kbd> move &nbsp; <kbd>N</kbd> note &nbsp; <kbd>Esc</kbd> leave note</p>
<script id="data" type="application/json">__DATA__</script>
<script id="meta" type="application/json">__META__</script>
<script>
const rows = JSON.parse(document.getElementById('data').textContent);
const meta = JSON.parse(document.getElementById('meta').textContent);
const $ = id => document.getElementById(id);

$('aboutBtn').onclick = () => {
  const open = $('about').hidden;
  $('about').hidden = !open;
  $('aboutBtn').setAttribute('aria-expanded', open);
  try { localStorage.setItem('docuscope-coding-about', open ? '1' : '0'); } catch (e) {}
};
try { if (localStorage.getItem('docuscope-coding-about') !== '0') { $('about').hidden = false; $('aboutBtn').setAttribute('aria-expanded', true); } } catch (e) { $('about').hidden = false; }
$('nPerRule').textContent = meta.n_per_rule;
$('nPerNoun').textContent = meta.n_per_noun;
$('nPerCat').textContent = meta.n_per_category;
$('seed').textContent = meta.random_state;
$('fixedRules').textContent = meta.fixed_rules.join(', ');
$('generated').textContent = `Generated ${meta.generated} from ${meta.emails} emails (${meta.rhetoric_tags} rhetorical tags, ${meta.authored_rhetoric_tags} in authored text).`;
let coder = '', codes = {}, view = [], pos = 0;

function storeKey() { return 'docuscope-coding-' + coder; }
function load() { try { codes = JSON.parse(localStorage.getItem(storeKey())) || {}; } catch (e) { codes = {}; } }
function save() { try { localStorage.setItem(storeKey(), JSON.stringify(codes)); } catch (e) {} }
function esc(s) { return String(s ?? '').replace(/[&<>"]/g, c => ({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;'}[c])); }

const samples = [...new Set(rows.map(r => r.sample))];
$('sampleTable').innerHTML = samples.map(s => `<tr><td><code>${esc(s)}</code></td><td>${rows.filter(r => r.sample === s).length === 1 ? '1 tag' : rows.filter(r => r.sample === s).length + ' tags'}</td></tr>`).join('');
$('filter').innerHTML = '<option value="">all samples</option>' +
  samples.map(s => `<option>${esc(s)}</option>`).join('');

function rebuild(keepId) {
  const f = $('filter').value, onlyUncoded = $('uncoded').checked;
  view = rows.filter(r => (!f || r.sample === f) && (!onlyUncoded || !(codes[r.id] && codes[r.id].code)));
  pos = Math.max(0, keepId != null ? view.findIndex(r => r.id === keepId) : 0);
  render();
}

function render() {
  const done = rows.filter(r => codes[r.id] && codes[r.id].code).length;
  $('count').textContent = `${done} / ${rows.length} coded`;
  $('bar').style.width = (100 * done / rows.length) + '%';
  if (!coder) { $('main').innerHTML = '<div class="empty">Choose a coder to start.</div>'; return; }
  if (!view.length) { $('main').innerHTML = '<div class="empty">Nothing to show with these filters.</div>'; return; }
  pos = Math.min(Math.max(pos, 0), view.length - 1);
  const r = view[pos], c = codes[r.id] || {};
  const cat = meta.categories[r.tag];
  const ruleList = r.rules ? r.rules.split(';') : [];
  const why = (ruleList.length ? ruleList : ['no_rule'])
    .map(k => `<div><b>${k === 'no_rule' ? 'No rule' : 'Rule ' + esc(k)}</b>: ${esc(meta.rule_descriptions[k] || '')}</div>`).join('');
  $('main').innerHTML = `
    <div class="card">
      <div class="meta"><span class="pill">${esc(r.stratum)}</span><span class="pill">${Number(r.stratum_size).toLocaleString()} tags in stratum</span><span class="pill">msgid2 ${esc(r.msgid2)}</span></div>
      <div class="tag">${esc(r.tag)}</div>
      ${cat ? `<div class="def">${esc(cat.desc)} <span class="eg">e.g. ${cat.eg.split('; ').map(x => '“' + esc(x) + '”').join(', ')}</span></div>` : ''}
      <div class="phrase">tagged phrase: “${esc(r.phrase)}”</div>
      <div class="why">${why}</div>
      <div class="context">…${esc(r.left)} <mark>${esc(r.phrase)}</mark> ${esc(r.right)}…</div>
      <div class="codes">
        ${['correct','incorrect','unsure'].map((k,i) =>
          `<button data-code="${k}" class="${c.code===k?'on':''}">${i+1} · ${k}</button>`).join('')}
      </div>
      <input class="note" id="note" type="text" placeholder="Note (optional)" value="${esc(c.note)}">
      <div class="nav"><button id="prev">← previous</button><span>${pos+1} of ${view.length}</span><button id="next">next →</button></div>
    </div>`;
  document.querySelectorAll('.codes button').forEach(b => b.onclick = () => setCode(b.dataset.code));
  $('prev').onclick = () => { pos--; render(); };
  $('next').onclick = () => { pos++; render(); };
  $('note').oninput = e => { const id = view[pos].id; codes[id] = {...(codes[id]||{}), note: e.target.value}; save(); };
}

function setCode(k) {
  const r = view[pos];
  codes[r.id] = {...(codes[r.id]||{}), code: k};
  save();
  if ($('uncoded').checked) { view.splice(pos, 1); } else { pos++; }
  render();
}

document.addEventListener('keydown', e => {
  if (!coder || !view.length) return;
  if (e.target.id === 'note') { if (e.key === 'Escape' || e.key === 'Enter') e.target.blur(); return; }
  if (e.target.tagName === 'SELECT' || e.target.tagName === 'INPUT') return;
  if (e.key === '1') setCode('correct');
  else if (e.key === '2') setCode('incorrect');
  else if (e.key === '3') setCode('unsure');
  else if (e.key === 'ArrowRight') { pos++; render(); }
  else if (e.key === 'ArrowLeft') { pos--; render(); }
  else if (e.key.toLowerCase() === 'n') { e.preventDefault(); $('note').focus(); }
});

$('coder').onchange = e => { coder = e.target.value; load(); rebuild(); };
$('filter').onchange = () => rebuild();
$('uncoded').onchange = () => rebuild();

function csvCell(v) { const s = String(v ?? ''); return /[",\n]/.test(s) ? '"' + s.replace(/"/g, '""') + '"' : s; }
$('export').onclick = () => {
  if (!coder) { alert('Choose a coder first.'); return; }
  const cols = ['id','sample','stratum','stratum_size','msgid2','tag','phrase','rules'];
  const lines = [[...cols, coder, coder + '_note'].join(',')].concat(rows.map(r =>
    [...cols.map(k => csvCell(r[k])), csvCell((codes[r.id]||{}).code), csvCell((codes[r.id]||{}).note)].join(',')));
  const a = document.createElement('a');
  a.href = URL.createObjectURL(new Blob([lines.join('\n') + '\n'], {type: 'text/csv'}));
  a.download = `docuscope_coding_${coder}.csv`;
  a.click();
};

function parseCSV(text) {
  const out = []; let row = [], cell = '', q = false;
  for (let i = 0; i < text.length; i++) {
    const ch = text[i];
    if (q) { if (ch === '"' && text[i+1] === '"') { cell += '"'; i++; } else if (ch === '"') q = false; else cell += ch; }
    else if (ch === '"') q = true;
    else if (ch === ',') { row.push(cell); cell = ''; }
    else if (ch === '\n') { row.push(cell); out.push(row); row = []; cell = ''; }
    else if (ch !== '\r') cell += ch;
  }
  if (cell || row.length) { row.push(cell); out.push(row); }
  return out;
}
$('import').onclick = () => { if (!coder) { alert('Choose a coder first.'); return; } $('file').click(); };
$('file').onchange = async e => {
  const f = e.target.files[0]; if (!f) return;
  const [head, ...body] = parseCSV(await f.text());
  const iId = head.indexOf('id'), iCode = head.indexOf(coder), iNote = head.indexOf(coder + '_note');
  if (iId < 0 || iCode < 0) { alert(`That file has no ${coder} column.`); return; }
  body.forEach(r => { if (r[iCode] || r[iNote]) codes[r[iId]] = {code: r[iCode] || undefined, note: iNote >= 0 ? r[iNote] : ''}; });
  save(); rebuild(); e.target.value = '';
};

render();
</script>
</body></html>
'''
tool_rows = coding[['id', 'sample', 'stratum', 'stratum_size', 'msgid2', 'tag', 'phrase',
                    'left', 'right', 'rules']].astype(str).to_dict('records')
tool_json = json.dumps(tool_rows, ensure_ascii=False).replace('</', '<\\/')
tool_meta = dict(generated=pd.Timestamp.now().strftime('%Y-%m-%d %H:%M'), emails=f"{len(pool9):,}",
                 rhetoric_tags=f"{len(how9):,}", authored_rhetoric_tags=f"{len(auth):,}",
                 n_per_rule=n_per_rule, n_per_noun=n_per_noun, n_per_category=n_per_category,
                 random_state=42, fixed_rules=sorted(fixed_rules),
                 categories={k: {'desc': d, 'eg': eg} for k, (d, eg) in category_descriptions.items()},
                 rule_descriptions=rule_descriptions)
tool_path = here / 'docuscope_coding_tool.html'
with open(tool_path, 'w', encoding='utf-8') as f:
    f.write(coding_html.replace('__DATA__', tool_json).replace('__META__', json.dumps(tool_meta)))
print(f"Coding tool saved to: {tool_path}")

##### 4 - Preliminary insights across tag versions

Reads `rhetoric_tags.parquet` from block 3 and compares four versions of the rhetorical tags: raw (all zones, as `rhetoric.ipynb` block 2 counted them), authored (the authored zone only), cleaned (authored, without tags that fixed rules flag) and upper bound (authored, without tags that any rule flags; provisional until coding confirms the checked rules). Reports each category's share in each version, the phrases that carry each category and how concentrated they are, and the category shares of CITF senders and study-team senders. A sender counts as CITF if their address is on a CITF domain or their emails span five or more study IDs. This rule is provisional, so check the domain table before reading the group results. Run it after block 3, since it reuses `pool9` and `fixed_rules`.

In [ ]:
import os
import re
import numpy as np
import pandas as pd

# Print tables as text so the full report is one copyable output
def show(df):
    print(df.to_string(max_colwidth=120))

# ---- Settings (edit and rerun) ----

data_dir = here

# Sender groups. A sender counts as CITF if their address is on a CITF domain, or if their emails
# carry at least min_studies distinct study IDs (CITF staff write to many studies; study teams to few).
# This is a provisional rule: check the domain tables below and adjust before reading the group results.
citf_domains = r"covid19immunitytaskforce\.ca"
min_studies = 5

# Phrases shown per category in the phrase tables
top_phrases = 5

# Rules applied without coding, as defined in block 3
fixed = set(fixed_rules)

# ---- Four versions of the rhetorical tags ----
# raw: all zones (what rhetoric.ipynb block 2 counted)
# authored: the authored zone only
# cleaned: authored, without tags that fixed rules flag
# upper bound: authored, without tags that any rule flags (provisional until coding confirms the checked rules)
tags = pd.read_parquet(os.path.join(data_dir, 'rhetoric_tags.parquet'))
tags['msgid2'] = tags['msgid2'].astype(str)
rule_sets = tags['rules'].fillna('').str.split(';').apply(lambda r: {x for x in r if x})
tags['fixed_hit'] = rule_sets.apply(lambda r: bool(r & fixed))
tags['any_hit'] = rule_sets.apply(bool)
is_auth = tags['zone'] == 'authored'
versions = {
    'raw': tags,
    'authored': tags[is_auth],
    'cleaned': tags[is_auth & ~tags['fixed_hit']],
    'upper bound': tags[is_auth & ~tags['any_hit']],
}
print("Rhetorical tags and emails in each version:")
show(pd.DataFrame({v: {'rhetorical tags': len(d), 'emails': d['msgid2'].nunique()} for v, d in versions.items()}).T)

# ---- 1. Comparison with earlier iterations: category shares ----
share = pd.DataFrame({v: d['tag'].value_counts(normalize=True).mul(100) for v, d in versions.items()}).fillna(0)
rank = share.rank(ascending=False, method='min').astype(int)
comp = share.round(1)
comp['raw to cleaned (points)'] = (share['cleaned'] - share['raw']).round(1)
comp['raw to cleaned (% of raw share)'] = (100 * (share['cleaned'] - share['raw']) / share['raw']).round(0)
comp['rank: raw, cleaned'] = rank['raw'].astype(str) + ', ' + rank['cleaned'].astype(str)
print("\n1. Share of rhetorical tags per category (%), per version:")
show(comp.sort_values('raw', ascending=False))

# ---- 2. The language behind the tags ----
def phrase_profile(d):
    rows = {}
    for c, s in d.assign(p=d['phrase'].str.lower()).groupby('tag')['p']:
        vc = s.value_counts()
        rows[c] = {'tags': len(s), 'distinct phrases': len(vc),
                   'top 10 phrases cover (%)': round(100 * vc.head(10).sum() / len(s), 1),
                   'top phrases': '; '.join(f"{k} ({v})" for k, v in vc.head(top_phrases).items())}
    return pd.DataFrame(rows).T

raw_p, clean_p = phrase_profile(versions['raw']), phrase_profile(versions['cleaned'])
order = raw_p['tags'].astype(int).sort_values(ascending=False).index
conc = pd.DataFrame({'distinct phrases, raw': raw_p['distinct phrases'],
                     'distinct phrases, cleaned': clean_p['distinct phrases'],
                     'top 10 cover, raw (%)': raw_p['top 10 phrases cover (%)'],
                     'top 10 cover, cleaned (%)': clean_p['top 10 phrases cover (%)']}).loc[order]
print("\n2a. How concentrated each category is: distinct phrases, and the share of tags the 10 most frequent phrases cover:")
show(conc)
print(f"\n2b. The {top_phrases} most frequent phrases per category (lowercased), raw and cleaned:")
show(pd.DataFrame({'raw': raw_p['top phrases'], 'cleaned': clean_p['top phrases']}).loc[order])

# ---- 3. CITF and study teams ----
em = pool9[['msgid2', 'from', 'msg_citfid']].copy()
em['msgid2'] = em['msgid2'].astype(str)
addr = em['from'].astype(str).str.lower()
em['sender'] = addr.str.extract(r'([\w.+\'-]+@[\w.-]+)')[0].fillna(addr)
em['domain'] = em['sender'].str.split('@').str[-1]
n_studies = em.groupby('sender')['msg_citfid'].nunique()
em['group'] = np.where(em['domain'].str.contains(citf_domains, regex=True)
                       | em['sender'].map(n_studies).ge(min_studies), 'CITF', 'study team')
print(f"\n3a. Sender groups (CITF: domain matches {citf_domains!r} or emails span {min_studies}+ study IDs):")
show(em.groupby('group').agg(emails=('msgid2', 'size'), senders=('sender', 'nunique')))
dom = (em.groupby(['group', 'domain']).agg(emails=('msgid2', 'size'), senders=('sender', 'nunique'))
       .reset_index().sort_values(['group', 'emails'], ascending=[True, False]).groupby('group').head(10))
print("\nTop 10 sender domains in each group (check the assignment here):")
show(dom.set_index(['group', 'domain']))

missing = {'CITF', 'study team'} - set(em['group'])
if missing:
    print(f"\nNo emails in group(s) {sorted(missing)}: adjust citf_domains or min_studies before reading 3b.")
group_of = em.set_index('msgid2')['group']
gaps = {}
for v in ['raw', 'cleaned']:
    d = versions[v].assign(group=versions[v]['msgid2'].map(group_of))
    s = (pd.crosstab(d['tag'], d['group'], normalize='columns').mul(100)
         .reindex(columns=['CITF', 'study team'], fill_value=0))
    gaps[f'CITF, {v}'] = s['CITF']
    gaps[f'study team, {v}'] = s['study team']
    gaps[f'gap, {v}'] = s['CITF'] - s['study team']
gap = pd.DataFrame(gaps)
big = gap['gap, raw'].abs() >= 0.5
gap['change'] = np.select(
    [big & (np.sign(gap['gap, raw']) != np.sign(gap['gap, cleaned'])),
     big & (gap['gap, cleaned'].abs() < 0.5 * gap['gap, raw'].abs()),
     gap['gap, cleaned'].abs() >= 2 * gap['gap, raw'].abs().clip(lower=0.25)],
    ['gap reverses', 'gap halves or more', 'gap doubles or more'], default='')
print("\n3b. Share of each group's rhetorical tags per category (%), raw and cleaned; gap = CITF minus study team (points):")
show(gap.round(1).loc[order])